# Chạy model CDNMF

Notebook chạy CDNMF trên Kaggle hoặc từ thư mục repository. Chạy `prepare_artist_cdnmf.ipynb` trước nếu dùng dữ liệu CSV; notebook đó tạo `Database/artist_csb`. Trên Kaggle, bảo đảm thư mục dữ liệu đã tạo được đưa vào thư mục repository đang chạy.

Bật **Settings → Internet** nếu cần tải repository/thư viện; chọn **Settings → Accelerator → GPU** nếu muốn dùng GPU. Chạy các cell theo thứ tự. Chọn `artist_csb` trong cell cấu hình để chạy trên dữ liệu nghệ sĩ; lần đầu notebook tự tạo tham số pretrain. Có thể giảm `RUNS` và `EPOCHS` để chạy thử nhanh.

Notebook giữ nguyên PyTorch có sẵn trên Kaggle để tránh cài đè bản CUDA. Các cặp trong `walks.txt` được dùng làm hạng smoothness bổ sung cho embedding cộng đồng; với Artist, chạy `prepare_artist_cdnmf.ipynb` để tạo random walks trước.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/HongHa-1606/CDNMF.git"
KAGGLE_WORKING = Path("/kaggle/working")
REPO_DIR = KAGGLE_WORKING / "CDNMF" if KAGGLE_WORKING.exists() else Path.cwd()

if not (REPO_DIR / "Database").is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise FileExistsError(
            f"Thư mục đích đã tồn tại nhưng không phải repository CDNMF: {REPO_DIR}"
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )

if not (REPO_DIR / "requirements.txt").is_file():
    raise FileNotFoundError(f"Không tìm thấy requirements.txt trong {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# Kaggle already provides CUDA-enabled PyTorch; do not replace it with the pinned CPU/runtime-specific version.
dependencies = {
    "numpy": "numpy",
    "networkx": "networkx",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "scipy": "scipy",
    "tqdm": "tqdm",
}
missing = [
    distribution
    for distribution, module in dependencies.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        check=True,
    )

import torch
print(f"Repository: {REPO_DIR}")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if missing:
    print(f"Installed missing dependencies: {', '.join(missing)}")
else:
    print("All required non-PyTorch packages are already installed.")

In [ ]:
from pathlib import Path
from copy import deepcopy
import random
import pickle
import tempfile

import numpy as np
import torch

from Dataset.dataset import Dataset
from Model.my_model import Model
from Utils.evaluate import clusterscores

ROOT = Path.cwd()
if not (ROOT / "Database").is_dir():
    raise FileNotFoundError(
        "Mở notebook với thư mục làm việc là thư mục gốc CDNMF (có thư mục Database)."
    )

DATASET = "cora"  # Chọn: "cora", "citeseer", "pubmed", "artist_csb"
RUNS = 20
EPOCHS = None  # Có thể giảm số epoch khi chạy thử

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Dataset: {DATASET} | Device: {device}")

In [ ]:
from PreTrainer.pretrainer import PreTrainer

dataset_settings = {
    "cora": {
        "net_shape": [1000, 500, 7],
        "att_shape": [500, 200, 7],
        "tau": 1.3,
        "conc": 5,
        "negc": 400,
        "r": 3,
        "walk_weight": 1.0,
        "learning_rate": 0.01,
        "epoch": 550,
        "pretrain_params_path": "Log/cora/pretrain_params.pkl",
    },
    "citeseer": {
        "net_shape": [256, 64, 6],
        "att_shape": [500, 100, 6],
        "tau": 1.5,
        "conc": 5,
        "negc": 3000,
        "r": 1,
        "walk_weight": 1.0,
        "learning_rate": 0.001,
        "epoch": 1000,
        "pretrain_params_path": "Log/citeseer/pretrain_params.pkl",
    },
    "pubmed": {
        "net_shape": [512, 64, 3],
        "att_shape": [200, 100, 3],
        "tau": 0.5,
        "conc": 1,
        "negc": 100,
        "r": 1,
        "walk_weight": 1.0,
        "learning_rate": 0.01,
        "epoch": 600,
        "pretrain_params_path": "Log/pubmed/pretrain_params.pkl",
    },
    "artist_csb": {
        "net_shape": [256, 64, None],
        "att_shape": [256, 64, None],
        "tau": 1.3,
        "conc": 5,
        "negc": 400,
        "r": 1,
        "walk_weight": 1.0,
        "learning_rate": 0.001,
        "epoch": 300,
        "pretrain_params_path": "Log/artist_csb/pretrain_params.pkl",
    },
}

if DATASET not in dataset_settings:
    raise ValueError(f"Dataset không hợp lệ: {DATASET}. Chọn một trong {list(dataset_settings)}.")

settings = dataset_settings[DATASET]
dataset_dir = ROOT / "Database" / DATASET
pretrain_path = ROOT / settings["pretrain_params_path"]
for required_file in ("edges.txt", "features.txt", "group.txt"):
    if not (dataset_dir / required_file).is_file():
        raise FileNotFoundError(
            f"Không tìm thấy {dataset_dir / required_file}. "
                "Hãy chạy prepare_artist_cdnmf.ipynb trước và đặt thư mục Database/artist_csb ở đây."
        )

walks_path = dataset_dir / "walks.txt"
walks_temp_dir = None
if not walks_path.is_file():
    walks_temp_dir = tempfile.TemporaryDirectory(prefix="cdnmf_walks_")
    walks_path = Path(walks_temp_dir.name) / "walks.txt"
    walks_path.write_text("", encoding="utf-8")

graph = Dataset({
    "feature_file": str(dataset_dir / "features.txt"),
    "graph_file": str(dataset_dir / "edges.txt"),
    "walks_file": str(walks_path),
    "label_file": str(dataset_dir / "group.txt"),
    "device": device,
})
if settings["walk_weight"] > 0 and graph.W.count_nonzero().item() == 0:
    raise ValueError(
        f"{walks_path} không có cặp walk. Hãy chạy lại prepare_artist_cdnmf.ipynb "
        "hoặc kiểm tra walks.txt của dataset."
    )
print(f"Walk pairs loaded: {int(graph.W.count_nonzero().item()) // 2:,}")

if DATASET == "artist_csb":
    settings["net_shape"][-1] = graph.num_classes
    settings["att_shape"][-1] = graph.num_classes
    if not pretrain_path.is_file():
        pretrain_path.parent.mkdir(parents=True, exist_ok=True)
        pretrainer = PreTrainer({
            "net_shape": settings["net_shape"],
            "att_shape": settings["att_shape"],
            "seed": 42,
            "pre_iterations": 100,
            "pretrain_params_path": str(pretrain_path),
        })
        print("Pretraining graph and feature views; this may take a while.")
        pretrainer.pre_training(graph.A.detach().cpu().numpy(), "net")
        pretrainer.pre_training(graph.X.t().detach().cpu().numpy(), "att")
        print(f"Saved pretrain parameters: {pretrain_path}")
elif not pretrain_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy tham số pretrain: {pretrain_path}")

model_config = {
    "device": device,
    "net_shape": settings["net_shape"],
    "att_shape": settings["att_shape"],
    "net_input_dim": graph.num_nodes,
    "att_input_dim": graph.num_feas,
    "is_init": True,
    "pretrain_params_path": str(pretrain_path),
    "tau": settings["tau"],
    "conc": settings["conc"],
    "negc": settings["negc"],
    "rec": 1,
    "r": settings["r"],
    "walk_weight": settings["walk_weight"],
    "learning_rate": settings["learning_rate"],
    "weight_decay": 0.00001,
    "epoch": settings["epoch"],
    "run": RUNS,
    "model_path": str(ROOT / "Log" / DATASET / f"{DATASET}_model.pkl"),
}
epochs = model_config["epoch"] if EPOCHS is None else EPOCHS
if RUNS < 1 or epochs < 1:
    raise ValueError("RUNS và EPOCHS phải lớn hơn hoặc bằng 1.")
print(f"Bắt đầu huấn luyện: {RUNS} lần, {epochs} epoch/lần")

In [ ]:
INIT_NOISE = 0.05
with open(pretrain_path, "rb") as handle:
    base_U, base_V = pickle.load(handle)

run_results = []
for run_index in range(RUNS):
    seed = 42 + run_index
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    rng = np.random.default_rng(seed)
    run_U, run_V = deepcopy(base_U), deepcopy(base_V)
    for params in (run_U, run_V):
        for name, value in params.items():
            noise = rng.normal(0, INIT_NOISE, value.shape).astype(np.float32)
            params[name] = np.maximum(value * np.exp(noise), 1e-8)

    run_pretrain_path = Path(tempfile.gettempdir()) / f"cdnmf_{DATASET}_pretrain_{seed}.pkl"
    with open(run_pretrain_path, "wb") as handle:
        pickle.dump([run_U, run_V], handle, protocol=pickle.HIGHEST_PROTOCOL)
    run_config = {**model_config, "pretrain_params_path": str(run_pretrain_path)}
    model = Model(run_config).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=run_config["learning_rate"],
    )

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        model()
        loss, loss_graph, loss_features, loss_contrastive, loss_smoothness, loss_nonnegative = model.loss(graph)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        embeddings = model().detach().cpu().numpy()
    y_pred = np.argmax(embeddings, axis=0)
    y_true = graph.L.detach().cpu().numpy()
    scores = clusterscores(y_pred, y_true)
    run_results.append(scores)
    print(
        f"Run {run_index + 1}/{RUNS} (seed={seed}): "
        f"loss={loss.item():.4f}, ACC={scores['ACC']:.4f}, "
        f"NMI={scores['NMI']:.4f}, clusters={np.unique(y_pred).size}, "
        f"loss_parts(A/X/contrast/smooth/nonneg)="
        f"{loss_graph.item():.1f}/{loss_features.item():.1f}/"
        f"{loss_contrastive.item():.1f}/{loss_smoothness.item():.1f}/"
        f"{loss_nonnegative.item():.1f}"
    )
    run_pretrain_path.unlink()

print("\n=== Kết quả trung bình ===")
for metric in ("ACC", "NMI", "ARI", "AMI"):
    values = [result[metric] for result in run_results]
    print(f"{metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")